# Feature selection — ReliefF

ReliefF đa lớp (Kononenko, 1994): với mỗi mẫu, lấy `RELIEF_K` near-hit cùng lớp và `RELIEF_K` near-miss của từng lớp khác (trọng số theo prior P(C)/(1−P(lớp mẫu))); khoảng cách Manhattan với diff = |a−b| / (max−min) của gene. Cài đặt NumPy vector hoá, dùng toàn bộ mẫu train làm mẫu tham chiếu. Lấy top `N_FEATURES` theo trọng số.

- **Đầu vào:** `DATA_DIR/fold_{1..5}/train.h5` (chỉ dùng tập **train**; val/test không đụng tới). Dữ liệu giữ nguyên log2 expression, không chuẩn hoá.
- **Đổi dataset:** sửa `DATA_DIR` ở cell cấu hình. Số gene chọn: `N_FEATURES` (mặc định 100).
- **Đầu ra** (`/kaggle/working/feature_selection/relieff/<dataset>/`):
  - `selected_genes/fold_k.csv`, `selected_genes_all_folds.csv`, `selected_genes.json` — gene được chọn theo fold (rank, score)
  - `metrics_per_fold.csv`, `metrics_summary.csv` — thời gian, CPU utilization, peak memory, energy, EDP, carbon footprint
- **Đo lường:** chỉ bao quanh lời gọi `select_features` (không tính thời gian đọc file). Energy/Carbon ước lượng bằng `codecarbon` (tracking_mode="process"); EDP = Energy(J) × Time(s).

In [ ]:
# tables: backend cho pd.read_hdf | codecarbon: do nang luong/CO2 | psutil: CPU & RAM
get_ipython().system("pip install -q codecarbon psutil tables")

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (vd .../h5/tcga_luad) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
N_FEATURES = 100          # so gene giu lai moi fold
N_FOLDS = 5
SEED = 42
METHOD = "relieff"
RELIEF_K = 10               # so near-hit / near-miss moi lop (mac dinh skrebate)
METHOD_PARAMS = {"n_neighbors": RELIEF_K}
# None -> codecarbon tu dinh vi (can Internet); hoac dat ma ISO 3 ky tu, vd "USA", "VNM"
COUNTRY_ISO_CODE = None

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/feature_selection", METHOD, DATASET_NAME)
os.makedirs(os.path.join(RESULT_DIR, "selected_genes"), exist_ok=True)

assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", "train.h5")), \
    f"Khong thay {DATA_DIR}/fold_1/train.h5 -- da Add Input dataset chua?"
print("Dataset :", DATASET_NAME, "->", DATA_DIR)
print("Method  :", METHOD, "| N_FEATURES =", N_FEATURES)
print("Ket qua :", RESULT_DIR)
print("So CPU  :", os.cpu_count())

## Đọc dữ liệu

In [ ]:
import json, time, threading, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}


def load_split(fold, split="train"):
    # File h5 (pandas, key="data"): index = sample_id, cot = ten gene + cot "label"
    df = pd.read_hdf(os.path.join(DATA_DIR, f"fold_{fold}", f"{split}.h5"), key="data")
    X = df.drop(columns="label")
    # Giu nguyen gia tri log2 expression, KHONG chuan hoa
    return X.values.astype(np.float64), df["label"].values.astype(int), X.columns.to_numpy(), df.index.to_numpy()


# Doc thu fold 1 de xem cau truc
X0, y0, genes0, ids0 = load_split(1)
print(f"fold_1/train: {X0.shape[0]} mau x {X0.shape[1]} gene | khoang gia tri [{X0.min():.2f}, {X0.max():.2f}]")
print("Vi du gene:", list(genes0[:5]), "| vi du sample:", list(ids0[:3]))
print("Phan bo lop:", pd.Series(y0).map(STAGE_NAMES).value_counts().to_dict())
del X0, y0, genes0, ids0

## Bộ đo tài nguyên

In [ ]:
import psutil

try:
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    _CODECARBON = True
except ImportError:
    _CODECARBON = False


class ResourceTracker:
    """Do tren 1 khoi lenh: thoi gian (s), CPU utilization, peak memory (RSS, gom ca
    tien trinh con cua joblib), energy (kWh, codecarbon), carbon (kg CO2eq) va
    Energy-Delay Product EDP = Energy(J) * Time(s)."""

    def __init__(self, name, interval=0.05):
        self.name, self.interval = name, interval
        self.proc = psutil.Process()
        self._stop = threading.Event()

    def _procs(self):
        try:
            return [self.proc] + self.proc.children(recursive=True)
        except psutil.Error:
            return [self.proc]

    def _poll(self):
        # RSS tong va CPU time cua worker con (tru phan CPU worker da dung truoc khi bat dau do)
        rss = 0
        for p in self._procs():
            try:
                rss += p.memory_info().rss
                if p.pid != self.proc.pid:
                    t = p.cpu_times()
                    self._child_cpu[p.pid] = t.user + t.system - self._child_base.get(p.pid, 0.0)
            except psutil.Error:
                pass
        self._peak_rss = max(self._peak_rss, rss)
        return rss

    def _sampler(self):
        while not self._stop.is_set():
            self._poll()
            self._sys_cpu.append(psutil.cpu_percent(interval=None))
            self._stop.wait(self.interval)

    def __enter__(self):
        self._child_cpu, self._sys_cpu, self._peak_rss = {}, [], 0
        self._child_base = {}
        for p in self._procs()[1:]:
            try:
                t = p.cpu_times()
                self._child_base[p.pid] = t.user + t.system
            except psutil.Error:
                pass
        self.tracker = None
        if _CODECARBON:
            try:
                kw = dict(project_name=self.name, output_dir=RESULT_DIR, save_to_file=False,
                          log_level="error", measure_power_secs=1, tracking_mode="process")
                self.tracker = (OfflineEmissionsTracker(country_iso_code=COUNTRY_ISO_CODE, **kw)
                                if COUNTRY_ISO_CODE else EmissionsTracker(**kw))
                self.tracker.start()
            except Exception as e:
                print("[ResourceTracker] codecarbon loi:", e)
                self.tracker = None
        self.base_rss = self._poll()
        self._child_cpu = {}
        psutil.cpu_percent(interval=None)
        t = self.proc.cpu_times()
        self._cpu0 = t.user + t.system
        self._stop.clear()
        self._thread = threading.Thread(target=self._sampler, daemon=True)
        self._thread.start()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.time_sec = time.perf_counter() - self._t0
        t = self.proc.cpu_times()
        main_cpu = t.user + t.system - self._cpu0
        self._stop.set()
        self._thread.join(timeout=2)
        self._poll()
        cpu_sec = main_cpu + sum(self._child_cpu.values())

        self.energy_kwh, self.carbon_kg = float("nan"), float("nan")
        if self.tracker is not None:
            try:
                self.carbon_kg = float(self.tracker.stop() or 0.0)
                data = self.tracker.final_emissions_data
                self.energy_kwh = float(data.energy_consumed) if data is not None else float("nan")
            except Exception as e:
                print("[ResourceTracker] codecarbon stop loi:", e)

        n_cpu = os.cpu_count() or 1
        self.metrics = {
            "time_sec": self.time_sec,
            "cpu_time_sec": cpu_sec,
            # % tren tong nang luc may (100% = dung het n_cpu core)
            "cpu_util_percent": 100.0 * cpu_sec / (self.time_sec * n_cpu) if self.time_sec > 0 else float("nan"),
            "avg_cores_used": cpu_sec / self.time_sec if self.time_sec > 0 else float("nan"),
            "system_cpu_percent_mean": float(np.mean(self._sys_cpu)) if self._sys_cpu else float("nan"),
            "peak_memory_mb": self._peak_rss / 1024 ** 2,
            "peak_memory_increase_mb": (self._peak_rss - self.base_rss) / 1024 ** 2,
            "energy_kwh": self.energy_kwh,
            "energy_joule": self.energy_kwh * 3.6e6,
            "edp_joule_sec": self.energy_kwh * 3.6e6 * self.time_sec,
            "carbon_kg_co2eq": self.carbon_kg,
            "carbon_g_co2eq": self.carbon_kg * 1e3,
        }
        return False

## Thuật toán: ReliefF

In [ ]:
def top_k(scores, k):
    scores = np.nan_to_num(np.asarray(scores, dtype=float), nan=-np.inf)
    idx = np.argsort(-scores, kind="stable")[:k]
    return idx, scores[idx]

def _scaled_diff_matrix(X):
    # diff(f, a, b) = |a - b| / range(f); gene hang so -> diff = 0
    rng = X.max(0) - X.min(0)
    rng[rng == 0] = np.inf
    Xs = X / rng
    n = len(Xs)
    D = np.empty((n, n))
    for i in range(n):
        D[i] = np.abs(Xs - Xs[i]).sum(1)
    return Xs, D


def relieff_scores(X, y, k=10):
    Xs, D = _scaled_diff_matrix(X)
    n, p = Xs.shape
    classes, counts = np.unique(y, return_counts=True)
    prior = dict(zip(classes, counts / n))
    W = np.zeros(p)
    for i in range(n):
        d = D[i].copy()
        d[i] = np.inf
        ci = y[i]
        hits = np.flatnonzero(y == ci)
        hits = hits[hits != i]
        if len(hits):
            hits = hits[np.argsort(d[hits], kind="stable")[:k]]
            W -= np.abs(Xs[hits] - Xs[i]).mean(0)
        for c in classes:
            if c == ci:
                continue
            miss = np.flatnonzero(y == c)
            miss = miss[np.argsort(d[miss], kind="stable")[:k]]
            W += prior[c] / (1 - prior[ci]) * np.abs(Xs[miss] - Xs[i]).mean(0)
    return W / n


def select_features(X, y, k):
    return top_k(relieff_scores(X, y, RELIEF_K), k)

## Chạy trên 5 fold

In [ ]:
all_rows, metric_rows = [], []

for fold in range(1, N_FOLDS + 1):
    X, y, genes, _ = load_split(fold, "train")   # CHI dung tap train cho feature selection
    k = min(N_FEATURES, X.shape[1])

    with ResourceTracker(f"{METHOD}_{DATASET_NAME}_fold{fold}") as rt:
        idx, scores = select_features(X, y, k)

    sel = pd.DataFrame({"fold": fold, "rank": np.arange(1, len(idx) + 1),
                        "gene": genes[idx], "score": scores})
    sel.to_csv(os.path.join(RESULT_DIR, "selected_genes", f"fold_{fold}.csv"), index=False)
    all_rows.append(sel)

    m = {"dataset": DATASET_NAME, "method": METHOD, "fold": fold,
         "n_train": X.shape[0], "n_genes_total": X.shape[1], "n_selected": len(idx), **rt.metrics}
    metric_rows.append(m)
    print(f"fold {fold}: {X.shape} -> {len(idx)} gene | {m['time_sec']:.2f}s | CPU {m['cpu_util_percent']:.1f}% "
          f"({m['avg_cores_used']:.2f} core) | peak {m['peak_memory_mb']:.0f} MB | "
          f"{m['energy_kwh']:.3e} kWh | {m['carbon_g_co2eq']:.4f} gCO2eq | top5: {list(sel.gene[:5])}")
    del X, y

selected = pd.concat(all_rows, ignore_index=True)
selected.to_csv(os.path.join(RESULT_DIR, "selected_genes_all_folds.csv"), index=False)
with open(os.path.join(RESULT_DIR, "selected_genes.json"), "w") as f:
    json.dump({f"fold_{r}": g.gene.tolist() for r, g in selected.groupby("fold")}, f, indent=1)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(RESULT_DIR, "metrics_per_fold.csv"), index=False)
metrics

## Tổng hợp độ đo

In [ ]:
cols = ["time_sec", "cpu_time_sec", "cpu_util_percent", "avg_cores_used", "system_cpu_percent_mean",
        "peak_memory_mb", "peak_memory_increase_mb", "energy_kwh", "energy_joule",
        "edp_joule_sec", "carbon_kg_co2eq", "carbon_g_co2eq"]
summary = metrics[cols].agg(["mean", "std", "min", "max"]).T
summary.loc["time_sec", "total"] = metrics["time_sec"].sum()
summary.loc["energy_kwh", "total"] = metrics["energy_kwh"].sum()
summary.loc["carbon_kg_co2eq", "total"] = metrics["carbon_kg_co2eq"].sum()
summary.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"))

with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "method": METHOD, "n_features": N_FEATURES,
               "n_folds": N_FOLDS, "seed": SEED, "params": METHOD_PARAMS, "cpu_count": os.cpu_count(),
               "country_iso_code": COUNTRY_ISO_CODE}, f, indent=1)
summary

## Độ ổn định tập gene giữa các fold (tham khảo)

In [ ]:
# Tham khao nhanh: do on dinh tap gene giua cac fold (Jaccard cap doi, trung binh)
sets = {r: set(g.gene) for r, g in selected.groupby("fold")}
folds = sorted(sets)
jac = pd.DataFrame(index=folds, columns=folds, dtype=float)
for a in folds:
    for b in folds:
        jac.loc[a, b] = len(sets[a] & sets[b]) / len(sets[a] | sets[b])
pairs = [jac.loc[a, b] for i, a in enumerate(folds) for b in folds[i + 1:]]
print(f"Jaccard trung binh giua cac fold: {np.mean(pairs):.3f}")
freq = selected.gene.value_counts()
print(f"Gene xuat hien o ca {N_FOLDS} fold: {int((freq == N_FOLDS).sum())}")
freq.rename("n_folds").to_csv(os.path.join(RESULT_DIR, "gene_frequency.csv"))
jac.round(3)

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/fs_{METHOD}_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path)
for root, _, files in os.walk(RESULT_DIR):
    for fn in sorted(files):
        print(" ", os.path.relpath(os.path.join(root, fn), RESULT_DIR))